# Chapter 5: Inference — Generating Text from Images

*Build a Multimodal Model from Scratch*

---

In Chapters 1–4 we built and trained a complete Vision-Language Model (VLM).
Now it's time to *use* it.  Inference—generating text conditioned on an
image—sounds simple: just keep predicting the next token until you hit `<eos>`.
But there are important choices to make:

* **Which token do we pick at each step?** Greedy, top-k, or nucleus sampling?
* **How does the visual prefix flow through the decoder at generation time?**
* **How do we evaluate whether the generated text is any good?**

This final chapter covers all three, builds a clean `generate()` function
from scratch, compares sampling strategies, and finishes with a small
interactive caption loop on our synthetic test images.

## 5.1  Autoregressive Generation — How It Works

During *training*, the GPT sees the entire target sequence at once and
predicts all next tokens in parallel (this is the "teacher-forcing" trick).
During *inference*, we have **no** target sequence.  We generate it one token
at a time:

```
Step 0: input = [<sos>]               → model predicts next token → "a"
Step 1: input = [<sos>, "a"]          → model predicts next token → " "
Step 2: input = [<sos>, "a", " "]     → model predicts next token → "r"
...
Step N: input = [<sos>, ..., "e"]     → model predicts next token → <eos>
```

With a visual prefix, the sequence looks like:

```
[V1, V2, ..., V_N_img, <sos>, t1, t2, ..., t_k]
 ↑ visual tokens (from projection)               ↑ generated text tokens
```

The visual tokens are always prepended and never change; only the text tokens
grow step by step.

**Key insight**: at each step we run the *entire* sequence through the GPT,
but we only look at the *last* output position to decide the next token.
This is O(T²) total work — a limitation that attention-based KV-cache (not
covered here) addresses in production.

In [ ]:
import os, sys, math, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs('figures', exist_ok=True)
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 5.2  Model Definitions (Self-contained)

We restate the model components so this notebook runs standalone.
The code is identical to Chapter 4.

In [ ]:
# ── ViT ──────────────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class ViTBlock(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )
    def forward(self, x):
        h, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + h
        return x + self.ffn(self.norm2(x))

class ViTEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=8, in_channels=3,
                 embed_dim=128, depth=4, n_heads=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.blocks    = nn.ModuleList([ViTBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm      = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        tokens = self.patch_embed(x)
        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos_embed
        for blk in self.blocks:
            tokens = blk(tokens)
        return self.norm(tokens)[:, 1:, :]

# ── GPT ──────────────────────────────────────────────────────────────────────
class GPTBlock(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )
    def _causal_mask(self, T, device):
        return torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()
    def forward(self, x, n_visual=0):
        T = x.shape[1]
        mask = self._causal_mask(T, x.device)
        if n_visual > 0:
            mask[:n_visual, :]    = False
            mask[n_visual:, :n_visual] = False
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=mask)
        x = x + attn_out
        return x + self.ffn(self.norm2(x))

class GPTDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, depth=4, n_heads=4, max_seq=256):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(max_seq, embed_dim)
        self.blocks      = nn.ModuleList([GPTBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm        = nn.LayerNorm(embed_dim)
        self.lm_head     = nn.Linear(embed_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight
        self.max_seq = max_seq
    def forward(self, input_ids, visual_prefix=None):
        B, T = input_ids.shape
        device = input_ids.device
        n_visual = 0
        if visual_prefix is not None:
            n_visual = visual_prefix.shape[1]
            vpos = self.pos_embed(torch.arange(n_visual, device=device)).unsqueeze(0)
            tpos = self.pos_embed(
                torch.arange(n_visual, n_visual + T, device=device)).unsqueeze(0)
            x = torch.cat([visual_prefix + vpos,
                            self.token_embed(input_ids) + tpos], dim=1)
        else:
            pos = self.pos_embed(torch.arange(T, device=device)).unsqueeze(0)
            x   = self.token_embed(input_ids) + pos
        for blk in self.blocks:
            x = blk(x, n_visual=n_visual)
        x = self.norm(x)
        return self.lm_head(x), n_visual

# ── Projection + VLM ────────────────────────────────────────────────────────
class ProjectionMLP(nn.Module):
    def __init__(self, vision_dim, language_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_dim, language_dim), nn.GELU(),
            nn.Linear(language_dim, language_dim),
        )
    def forward(self, x):
        return self.net(x)

class VisionLanguageModel(nn.Module):
    def __init__(self, vit, projection, gpt):
        super().__init__()
        self.vit = vit; self.projection = projection; self.gpt = gpt
    def forward(self, images, input_ids, labels=None):
        patch_tokens  = self.vit(images)
        visual_prefix = self.projection(patch_tokens)
        logits, n_visual = self.gpt(input_ids, visual_prefix=visual_prefix)
        loss = None
        if labels is not None:
            text_logits  = logits[:, n_visual:-1, :]
            text_targets = labels[:, 1:]
            loss = F.cross_entropy(
                text_logits.reshape(-1, text_logits.size(-1)),
                text_targets.reshape(-1), ignore_index=-100)
        return logits, loss
    def set_stage1(self):
        for p in self.vit.parameters():        p.requires_grad_(False)
        for p in self.gpt.parameters():        p.requires_grad_(False)
        for p in self.projection.parameters(): p.requires_grad_(True)
    def set_stage2(self):
        for p in self.vit.parameters():        p.requires_grad_(False)
        for p in self.projection.parameters(): p.requires_grad_(True)
        for p in self.gpt.parameters():        p.requires_grad_(True)

print('All model classes defined.')

## 5.3  Quick Training Run

We re-run the full two-stage training in this notebook so the model we
do inference on was trained here.  If you already ran Chapter 4, you could
save and reload weights—but keeping everything in one notebook makes the
code easier to follow.

In [ ]:
CAPTIONS = ['a red image', 'a blue image', 'vertical stripes', 'horizontal stripes']

class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set(''.join(texts)))
        self.vocab = ['<pad>', '<sos>', '<eos>'] + chars
        self.stoi  = {c: i for i, c in enumerate(self.vocab)}
        self.itos  = {i: c for i, c in enumerate(self.vocab)}
        self.pad_id = 0; self.sos_id = 1; self.eos_id = 2
    def encode(self, text, max_len=32, add_special=True):
        ids = [self.stoi[c] for c in text if c in self.stoi]
        if add_special:
            ids = [self.sos_id] + ids + [self.eos_id]
        ids = ids[:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids
    def decode(self, ids):
        out = []
        for i in ids:
            tok = self.itos.get(i, '')
            if tok in ('<pad>', '<sos>'): continue
            if tok == '<eos>': break
            out.append(tok)
        return ''.join(out)
    @property
    def vocab_size(self): return len(self.vocab)

tokenizer = CharTokenizer(CAPTIONS)

class ImageCaptionDataset(Dataset):
    IMG_SIZE = 32
    def __init__(self, n_samples=800, tokenizer=None, max_len=24):
        self.tokenizer = tokenizer; self.max_len = max_len
        self.data = self._generate(n_samples)
    def _make_image(self, label):
        img = torch.zeros(3, 32, 32)
        if label == 0:   img[0] = 0.9
        elif label == 1: img[2] = 0.9
        elif label == 2:
            for c in range(32):
                img[0 if c%8<4 else 2, :, c] = 0.9
        else:
            for r in range(32):
                img[0 if r%8<4 else 2, r, :] = 0.9
        return img + torch.randn_like(img) * 0.05
    def _generate(self, n):
        data = []
        for i in range(n):
            label = i % 4
            img   = self._make_image(label)
            ids   = torch.tensor(
                self.tokenizer.encode(CAPTIONS[label], max_len=self.max_len),
                dtype=torch.long)
            data.append((img, ids, label))
        return data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

dataset    = ImageCaptionDataset(tokenizer=tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
print(f'Dataset: {len(dataset)} samples, vocab size: {tokenizer.vocab_size}')

In [ ]:
VIT_DIM = LANG_DIM = 128
MAX_LEN = 24

vit        = ViTEncoder(img_size=32, patch_size=8, embed_dim=VIT_DIM, depth=4, n_heads=4)
projection = ProjectionMLP(VIT_DIM, LANG_DIM)
gpt        = GPTDecoder(tokenizer.vocab_size, embed_dim=LANG_DIM,
                        depth=4, n_heads=4, max_seq=MAX_LEN + 16 + 4)
model = VisionLanguageModel(vit, projection, gpt).to(DEVICE)

def train_epoch(model, loader, opt):
    model.train()
    total = 0.0
    for imgs, caps, _ in loader:
        imgs, caps = imgs.to(DEVICE), caps.to(DEVICE)
        _, loss = model(imgs, caps, labels=caps)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item()
    return total / len(loader)

# Stage 1
model.set_stage1()
opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-3)
print('Stage 1 ...')
for epoch in range(20):
    loss = train_epoch(model, dataloader, opt1)
    if (epoch+1) % 5 == 0: print(f'  Epoch {epoch+1:3d}  loss={loss:.4f}')

# Stage 2
model.set_stage2()
opt2  = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=20, eta_min=1e-4)
print('Stage 2 ...')
for epoch in range(20):
    loss = train_epoch(model, dataloader, opt2)
    sched.step()
    if (epoch+1) % 5 == 0: print(f'  Epoch {epoch+1:3d}  loss={loss:.4f}')

print('Training complete.')

## 5.4  Sampling Strategies

At each generation step, the GPT produces a probability distribution over
the entire vocabulary.  We need a rule to pick one token.

### Greedy Decoding

Always pick the **argmax** — the single most probable token.

```
token = logits.argmax(dim=-1)
```

**Pros**: deterministic, reproducible, often gives the single most likely sequence.
**Cons**: can be repetitive; once the model is slightly wrong it has no way to recover.

### Temperature Scaling

Before taking argmax (or sampling), divide the logits by a temperature τ:

```
probs = softmax(logits / τ)
```

* τ = 1.0 → original distribution
* τ < 1.0 → sharper (more confident, more greedy-like)
* τ > 1.0 → flatter (more random, more creative)

### Top-k Sampling

Keep only the k most probable tokens, renormalize, then sample:

```python
top_k_logits, top_k_idx = logits.topk(k)
probs = F.softmax(top_k_logits, dim=-1)
next_token = top_k_idx[torch.multinomial(probs, 1)]
```

**Why it helps**: prevents the model from ever picking a very unlikely token
(which greedy may never choose, but simple temperature sampling might).

### Nucleus (Top-p) Sampling

Instead of a fixed k, keep the smallest set of tokens whose cumulative
probability ≥ p (e.g. p = 0.9):

```python
sorted_probs, sorted_idx = probs.sort(descending=True)
cumsum = sorted_probs.cumsum(dim=-1)
# remove tokens once cumulative > p
sorted_probs[cumsum - sorted_probs > p] = 0
```

**Why it helps**: the cutoff adapts to the distribution — when the model is
confident (one token has 95% probability) you don't waste time considering
the other 5%; when it's unsure, you keep more options open.

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, image, max_new_tokens=30,
             strategy='greedy', temperature=1.0, top_k=0, top_p=0.9):
    """
    Generate a caption for `image` using the specified decoding strategy.

    Args:
        model          : trained VisionLanguageModel
        tokenizer      : CharTokenizer
        image          : (1, C, H, W) tensor on DEVICE
        max_new_tokens : maximum tokens to generate
        strategy       : 'greedy' | 'temperature' | 'top_k' | 'nucleus'
        temperature    : for 'temperature' / 'top_k' / 'nucleus'
        top_k          : for 'top_k' (k value)
        top_p          : for 'nucleus' (p value, 0 < p <= 1)

    Returns:
        generated string
    """
    model.eval()
    device = image.device

    # 1. Encode image once; project to language space
    patch_tokens  = model.vit(image)                     # (1, N_patches, vit_dim)
    visual_prefix = model.projection(patch_tokens)       # (1, N_patches, lang_dim)

    # 2. Start with <sos>
    generated = [tokenizer.sos_id]

    for _ in range(max_new_tokens):
        input_ids = torch.tensor([generated], dtype=torch.long, device=device)

        # Forward pass
        logits, n_visual = model.gpt(input_ids, visual_prefix=visual_prefix)
        # Grab logits at the last text position
        next_logits = logits[0, -1, :].float()        # (vocab_size,)

        # 3. Decoding strategy
        if strategy == 'greedy':
            next_token = next_logits.argmax().item()

        elif strategy == 'temperature':
            scaled = next_logits / max(temperature, 1e-5)
            probs  = F.softmax(scaled, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

        elif strategy == 'top_k':
            scaled = next_logits / max(temperature, 1e-5)
            k = max(1, top_k)
            top_vals, top_idx = scaled.topk(k)
            probs  = F.softmax(top_vals, dim=-1)
            choice = torch.multinomial(probs, 1).item()
            next_token = top_idx[choice].item()

        elif strategy == 'nucleus':
            scaled = next_logits / max(temperature, 1e-5)
            probs  = F.softmax(scaled, dim=-1)
            sorted_probs, sorted_idx = probs.sort(descending=True)
            cumsum = sorted_probs.cumsum(dim=-1)
            # Remove tokens beyond the nucleus
            remove_mask = (cumsum - sorted_probs) > top_p
            sorted_probs[remove_mask] = 0.0
            sorted_probs /= sorted_probs.sum()
            choice = torch.multinomial(sorted_probs, 1).item()
            next_token = sorted_idx[choice].item()

        else:
            raise ValueError(f'Unknown strategy: {strategy}')

        generated.append(next_token)

        if next_token == tokenizer.eos_id:
            break

    return tokenizer.decode(generated)


print('generate() defined.')

## 5.5  Comparing Strategies on a Single Image

In [ ]:
def make_test_image(label):
    """Create a clean test image (no noise) for inference."""
    img = torch.zeros(1, 3, 32, 32)
    if label == 0:
        img[0, 0] = 0.9
    elif label == 1:
        img[0, 2] = 0.9
    elif label == 2:
        for c in range(32):
            img[0, 0 if c%8<4 else 2, :, c] = 0.9
    else:
        for r in range(32):
            img[0, 0 if r%8<4 else 2, r, :] = 0.9
    return img.to(DEVICE)


strategies = [
    ('greedy',      dict(strategy='greedy')),
    ('temp=0.5',    dict(strategy='temperature', temperature=0.5)),
    ('temp=1.5',    dict(strategy='temperature', temperature=1.5)),
    ('top_k=5',     dict(strategy='top_k', temperature=0.8, top_k=5)),
    ('nucleus=0.9', dict(strategy='nucleus', temperature=1.0, top_p=0.9)),
]

print(f'{"Class":<8} {"Ground truth":<22}', end='')
for name, _ in strategies:
    print(f'  {name:<14}', end='')
print()
print('-' * (8 + 22 + len(strategies) * 16))

for label in range(4):
    img = make_test_image(label)
    gt  = CAPTIONS[label]
    print(f'{label:<8} {gt:<22}', end='')
    for name, kwargs in strategies:
        out = generate(model, tokenizer, img, **kwargs)
        print(f'  {out:<14}', end='')
    print()

### Visualizing the Probability Distribution

The figure below shows, for one image, how each strategy selects the next
token from the same underlying logit distribution.

In [ ]:
def plot_strategy_comparison():
    img   = make_test_image(0)   # red image
    model.eval()
    with torch.no_grad():
        patch_tokens  = model.vit(img)
        visual_prefix = model.projection(patch_tokens)
        # Run with just <sos>
        input_ids = torch.tensor([[tokenizer.sos_id]], dtype=torch.long, device=DEVICE)
        logits, _ = model.gpt(input_ids, visual_prefix=visual_prefix)
        raw_logits = logits[0, -1, :].float().cpu()

    top_n = 10
    top_vals, top_idx = raw_logits.topk(top_n)
    top_tokens = [tokenizer.itos.get(i.item(), '?') for i in top_idx]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Original distribution
    probs_orig = F.softmax(top_vals, dim=-1).numpy()
    axes[0].bar(top_tokens, probs_orig, color='#60a5fa')
    axes[0].set_title('Original distribution\n(top 10 tokens)', fontweight='bold')
    axes[0].set_xlabel('Token'); axes[0].set_ylabel('Probability')

    # Low temperature (sharper)
    probs_cold = F.softmax(top_vals / 0.4, dim=-1).numpy()
    axes[1].bar(top_tokens, probs_cold, color='#4ade80')
    axes[1].set_title('Temperature = 0.4\n(sharper / more greedy)', fontweight='bold')
    axes[1].set_xlabel('Token')

    # High temperature (flatter)
    probs_hot = F.softmax(top_vals / 1.5, dim=-1).numpy()
    axes[2].bar(top_tokens, probs_hot, color='#f97316')
    axes[2].set_title('Temperature = 1.5\n(flatter / more random)', fontweight='bold')
    axes[2].set_xlabel('Token')

    plt.suptitle('Effect of Temperature on Token Probability Distribution',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/sampling_strategies.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_strategy_comparison()

## 5.6  Evaluating Generated Captions

How do we measure whether a generated caption is "good"?  We introduce
three common metrics.

### Exact Match Accuracy

The simplest metric: what fraction of generated captions exactly match the
ground truth?  Appropriate when the output space is small (like our 4 captions).

### BLEU (Bilingual Evaluation Understudy)

BLEU measures n-gram overlap between the generated caption and the reference.
BLEU-1 counts unigram matches; BLEU-2 counts bigram matches.

```
BLEU-1 = (matched unigrams) / (total unigrams in hypothesis)
```

Modified precision: count each reference n-gram at most as many times as it
appears in the reference (prevents reward for repeating one word many times).

BLEU is easy to compute and correlates reasonably with human judgment for
short captions, but it ignores word order (beyond n-grams) and synonyms.

### Perplexity

Perplexity measures how "surprised" the model is by the ground-truth caption:

```
PP = exp(cross_entropy_loss)
```

Lower perplexity = the model assigns higher probability to the correct text.
Perplexity is a model-internal metric — it tells you how well the model's
internal distribution matches the data, not how readable the output is.

In [ ]:
from collections import Counter

def bleu_n(hypothesis, reference, n=1):
    """Compute BLEU-n score for a single (hypothesis, reference) pair."""
    hyp_ngrams = [tuple(hypothesis[i:i+n]) for i in range(len(hypothesis)-n+1)]
    ref_ngrams = [tuple(reference[i:i+n]) for i in range(len(reference)-n+1)]
    if not hyp_ngrams: return 0.0

    ref_counts = Counter(ref_ngrams)
    clipped = sum(min(count, ref_counts[gram])
                  for gram, count in Counter(hyp_ngrams).items())
    return clipped / len(hyp_ngrams)


@torch.no_grad()
def compute_perplexity(model, image, caption_ids):
    """Return perplexity of caption_ids given image."""
    model.eval()
    img = image.unsqueeze(0).to(DEVICE)  # (1, C, H, W)
    ids = caption_ids.unsqueeze(0).to(DEVICE)
    _, loss = model(img, ids, labels=ids)
    return math.exp(loss.item())


def evaluate_model(model, tokenizer, dataset, n_eval=100):
    model.eval()
    exact_matches = 0
    bleu1_scores  = []
    bleu2_scores  = []
    perplexities  = []

    indices = random.sample(range(len(dataset)), min(n_eval, len(dataset)))

    for idx in indices:
        img, caps, label = dataset[idx]
        img = img.to(DEVICE)

        # Generate with greedy
        gen = generate(model, tokenizer, img.unsqueeze(0), strategy='greedy')
        ref = tokenizer.decode(caps.tolist())

        gen_chars = list(gen)
        ref_chars = list(ref)

        # Exact match
        if gen == ref:
            exact_matches += 1

        # BLEU
        bleu1_scores.append(bleu_n(gen_chars, ref_chars, n=1))
        if len(gen_chars) >= 2 and len(ref_chars) >= 2:
            bleu2_scores.append(bleu_n(gen_chars, ref_chars, n=2))

        # Perplexity
        perplexities.append(compute_perplexity(model, img, caps))

    n = len(indices)
    results = {
        'exact_match_accuracy': exact_matches / n,
        'bleu1':                np.mean(bleu1_scores),
        'bleu2':                np.mean(bleu2_scores) if bleu2_scores else 0.0,
        'mean_perplexity':      np.mean(perplexities),
        'n_evaluated':          n,
    }
    return results


results = evaluate_model(model, tokenizer, dataset)
print('=== Evaluation Results ===')
for k, v in results.items():
    print(f'  {k:<28}: {v:.4f}' if isinstance(v, float) else f'  {k:<28}: {v}')

In [ ]:
def per_class_evaluation(model, tokenizer, dataset):
    model.eval()
    per_class = {i: {'correct': 0, 'total': 0, 'bleu1': []} for i in range(4)}

    for img, caps, label in dataset:
        img   = img.to(DEVICE)
        gen   = generate(model, tokenizer, img.unsqueeze(0), strategy='greedy')
        ref   = tokenizer.decode(caps.tolist())
        per_class[label]['total'] += 1
        if gen.strip() == ref.strip():
            per_class[label]['correct'] += 1
        per_class[label]['bleu1'].append(bleu_n(list(gen), list(ref), n=1))

    print(f'{"Class":<6} {"Caption":<22} {"Accuracy":>10} {"BLEU-1":>8}')
    print('-' * 52)
    for cls in range(4):
        stats = per_class[cls]
        acc   = stats['correct'] / max(stats['total'], 1)
        b1    = np.mean(stats['bleu1'])
        print(f'{cls:<6} {CAPTIONS[cls]:<22} {acc:>10.2%} {b1:>8.4f}')

per_class_evaluation(model, tokenizer, dataset)

## 5.7  Visual Inference Demo

Let's run the full pipeline visually: show the image, the ground truth
caption, and what each strategy generates.

In [ ]:
def visual_inference_demo():
    fig, axes = plt.subplots(4, 2, figsize=(12, 14))

    for label in range(4):
        img = make_test_image(label)
        img_np = img[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()

        # Show image
        ax_img = axes[label][0]
        ax_img.imshow(img_np)
        ax_img.set_title(f'Class {label}: "{CAPTIONS[label]}"',
                         fontsize=10, fontweight='bold')
        ax_img.axis('off')

        # Show generation results
        ax_text = axes[label][1]
        ax_text.axis('off')

        lines = [f'Ground truth: "{CAPTIONS[label]}"\n']
        for strat, kwargs in [
            ('greedy',   dict(strategy='greedy')),
            ('top-k=5',  dict(strategy='top_k', temperature=0.8, top_k=5)),
            ('nucleus',  dict(strategy='nucleus', temperature=1.0, top_p=0.9)),
        ]:
            out = generate(model, tokenizer, img, **kwargs)
            match = '(correct)' if out.strip() == CAPTIONS[label] else '(wrong)'
            lines.append(f'{strat:>12}: "{out}" {match}')

        text = '\n'.join(lines)
        ax_text.text(0.02, 0.5, text, transform=ax_text.transAxes,
                     va='center', fontsize=9.5, family='monospace',
                     bbox=dict(boxstyle='round', facecolor='#f8fafc', alpha=0.8))

    plt.suptitle('VLM Inference: Strategies Compared', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/inference_demo.png', dpi=120, bbox_inches='tight')
    plt.show()

visual_inference_demo()

## 5.8  Token-by-Token Generation Trace

Let's peek inside the generation loop and watch probabilities change at
each step.  This shows how the model conditions each prediction on all
previously generated tokens.

In [ ]:
def generation_trace(model, tokenizer, image, max_steps=10):
    """Print the top-3 candidates at each decoding step."""
    model.eval()
    device = image.device
    with torch.no_grad():
        patch_tokens  = model.vit(image)
        visual_prefix = model.projection(patch_tokens)

    generated = [tokenizer.sos_id]
    print(f'Starting generation trace ...')
    print(f'{"Step":>4}  {"Context":<20}  {"Top-3 candidates (token : prob)"}')
    print('-' * 68)

    for step in range(max_steps):
        with torch.no_grad():
            input_ids = torch.tensor([generated], dtype=torch.long, device=device)
            logits, _ = model.gpt(input_ids, visual_prefix=visual_prefix)
            probs = F.softmax(logits[0, -1, :].float(), dim=-1)
            top3_probs, top3_idx = probs.topk(3)

        context = tokenizer.decode(generated)
        candidates = '  '.join(
            f'"{tokenizer.itos.get(i.item(), "?")}": {p:.3f}'
            for i, p in zip(top3_idx, top3_probs)
        )
        print(f'{step+1:>4}  {context!r:<20}  {candidates}')

        next_token = top3_idx[0].item()  # greedy
        generated.append(next_token)
        if next_token == tokenizer.eos_id:
            print(f'       <eos> generated, stopping.')
            break

    print(f'\nFinal: "{tokenizer.decode(generated)}"')


test_img = make_test_image(0)  # solid red
generation_trace(model, tokenizer, test_img)

In [ ]:
def plot_evaluation_summary(model, tokenizer, dataset):
    # Evaluate each strategy
    strategy_configs = [
        ('Greedy',       dict(strategy='greedy')),
        ('Top-k (k=5)',  dict(strategy='top_k', temperature=0.8, top_k=5)),
        ('Nucleus (0.9)',dict(strategy='nucleus', temperature=1.0, top_p=0.9)),
    ]

    names, accs, bleus = [], [], []

    for name, kwargs in strategy_configs:
        exact = 0; b1_total = []; n = 0
        for img, caps, label in random.sample(dataset.data, 80):
            img = img.to(DEVICE)
            gen = generate(model, tokenizer, img.unsqueeze(0), **kwargs)
            ref = tokenizer.decode(caps.tolist())
            if gen.strip() == ref.strip(): exact += 1
            b1_total.append(bleu_n(list(gen), list(ref), n=1))
            n += 1
        names.append(name)
        accs.append(exact / n)
        bleus.append(np.mean(b1_total))

    x = np.arange(len(names))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars1 = ax.bar(x - w/2, accs, w, label='Exact Match Accuracy',
                   color='#3b82f6', alpha=0.85)
    bars2 = ax.bar(x + w/2, bleus, w, label='BLEU-1 Score',
                   color='#22c55e', alpha=0.85)

    def label_bars(bars):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

    label_bars(bars1); label_bars(bars2)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
    ax.set_ylim(0, 1.2); ax.set_ylabel('Score')
    ax.set_title('Decoding Strategy Comparison', fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig('figures/evaluation_summary.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_evaluation_summary(model, tokenizer, dataset)

## 5.9  Chapter Summary

| Concept | Key Idea |
|---------|----------|
| **Autoregressive generation** | Generate one token at a time; feed all prior tokens back as input |
| **Visual prefix at inference** | ViT + Projection run once; visual tokens stay constant throughout generation |
| **Greedy decoding** | Fast, deterministic; tends to be repetitive |
| **Temperature scaling** | τ < 1 sharpens distribution; τ > 1 flattens it |
| **Top-k sampling** | Sample from the k most likely tokens; prevents tail noise |
| **Nucleus sampling** | Adaptive cutoff at cumulative probability p; best of both worlds |
| **BLEU** | N-gram overlap metric; quick proxy for caption quality |
| **Perplexity** | Model's own surprise at the reference; PP = exp(loss) |

---

## Book Summary

Congratulations! You have built a complete Vision-Language Model from scratch:

```
Chapter 1  →  Vision Transformer (ViT)
               PatchEmbedding (Conv2d trick) + bidirectional attention
               + learnable [CLS] token + positional embeddings

Chapter 2  →  CLIP
               Dual encoder + InfoNCE contrastive loss
               + learnable temperature + zero-shot classification

Chapter 3  →  VLM Architecture
               ProjectionMLP (semantic gap bridge)
               + GPT with visual_prefix parameter
               + loss masking (-100 for visual tokens)

Chapter 4  →  Two-Stage Training
               Stage 1: align projection only (cheap, fast)
               Stage 2: fine-tune projection + GPT (full power)
               + ablation study confirming both stages are needed

Chapter 5  →  Inference
               Autoregressive generation with visual prefix
               + greedy / temperature / top-k / nucleus decoding
               + BLEU and perplexity evaluation
```

Every component in this stack—ViT, CLIP, the projection bridge, the GPT
decoder, the two-stage training recipe, and the sampling algorithms—was
built directly inside notebook cells, no black boxes.

The same design decisions appear in production VLMs (LLaVA, InstructBLIP,
Flamingo, GPT-4V): the only difference is scale and data.
